In [2]:
from datetime import datetime
from time import sleep
import urllib.request
from bs4 import BeautifulSoup
import re
import pandas as pd
from pandas import Series, DataFrame

""" DFの最大出力行数を50から80へ """
pd.set_option("display.max_rows", 80)

cc = {}
cc['札幌'] = '01'
cc['函館'] = '02'
cc['福島'] = '03'
cc['新潟'] = '04'
cc['東京'] = '05'
cc['中山'] = '06'
cc['中京'] = '07'
cc['京都'] = '08'
cc['阪神'] = '09'
cc['小倉'] = '10'

rc = {}
rc['中山sat'] = cc['中山'] + '0403'
rc['中山sun'] = cc['中山'] + '0404'
rc['中山mon'] = cc['中山'] + '0405'
rc['阪神sat'] = cc['阪神'] + '0403'
rc['阪神sun'] = cc['阪神'] + '0404'
rc['阪神mon'] = cc['阪神'] + '0405'

def myEntriesDF( race_code):

    """ データ取得 """
    entries1_html = urllib.request.urlopen("https://keiba.yahoo.co.jp/race/denma/19" + race_code + "/?page=1")
    entries1_soup = BeautifulSoup(entries1_html,"lxml")
    df1 = pd.io.html.read_html(entries1_soup.prettify())
    entries1_df = df1[2]
    sleep(1)

    entries2_html = urllib.request.urlopen("https://keiba.yahoo.co.jp/race/denma/19" + race_code + "/?page=2")
    entries2_soup = BeautifulSoup(entries2_html,"lxml")
    df2 = pd.io.html.read_html(entries2_soup.prettify())
    entries2_df = df2[2]
    sleep(1)

    entries3_html = urllib.request.urlopen("https://keiba.yahoo.co.jp/race/denma/19" + race_code + "/?page=3")
    entries3_soup = BeautifulSoup(entries3_html,"lxml")
    df3 = pd.io.html.read_html(entries3_soup.prettify())
    entries3_df = df3[2]
    sleep(1)

    """ レース基本情報 """
    race_no = entries1_soup.find_all("td", attrs={'id':'raceNo'}) # <td id="raceNo">
    race_no = re.sub(r"[ \n\[\]]","",str(race_no))
    race_no = re.sub(r"<[^>]*?>","",str(race_no))
    # print(race_no)
    race_name = entries1_soup.find_all("h1",class_="fntB")
    race_name = re.sub(r"<[^>]*?>","",str(race_name))
    race_name = re.sub(r"[ \n\[\]]","",str(race_name))
    race_name = re.sub(r"—","-",str(race_name)) # 大阪-ハンブルグカップ対策
    race_name = re.sub(r"〜","-",str(race_name)) # 19XX-19XXsダービーメモリーズ対策
    race_name = re.sub(r"第.*?回","",str(race_name)) # 重賞の回次を削除
    # print(race_name)
    info_basic = entries1_soup.find_all("p",class_="fntSS",attrs={'id':'raceTitDay'})
    info_basic = re.sub(r"[ \n\[\]]","",str(info_basic))
    info_basic = re.sub(r"<[^>]*?>","",str(info_basic))
    info_basic = info_basic.split("|") # |をデリミタとしてリスト化
    start_time = info_basic.pop(2)
    info_basic.append(race_no)
    info_basic.append(race_name)
    info_basic.append(start_time)

    info_meta = entries1_soup.find_all("p",class_="fntSS gryB",attrs={'id':'raceTitMeta'})
    info_meta = re.sub(r"<img alt=""","",str(info_meta)) # 天気: 馬場:
    info_meta = re.sub(r" border=""","<",str(info_meta))# :雨 :良
    info_meta = re.sub(r"<[^>]*?>","",str(info_meta)) # <と中の^>（でない）文字の繰り返しと>
    info_meta = re.sub(r"\"","",str(info_meta)) # "雨" "良"の(")の削除
    info_meta = re.sub(r"[ \n\[\]]","",str(info_meta))
    info_meta = info_meta.split("|") # |をデリミタとしてリスト化
    #print(info_basic)
    #print(info_meta)
    info_race = info_basic + info_meta

    """ 1) 出走場基本情報 """
    HrsInfo = []
    for h in range(len(entries1_df)):
        horse_number = str(entries1_df.iloc[[h], [1]].values[0][0]).rjust(2,"0")
        horse_name = str(entries1_df.iloc[[h], [2]].values[0][0]).split(" ")[0]
        sex_age = re.sub("  "," ",str(entries1_df.iloc[[h], [2]].values[0][0])).split(" ")[1].split("/")[0]
        weight = str(entries1_df.iloc[[h], [3]].values[0][0]).replace("  ", "")
        jockey = str(entries1_df.iloc[[h], [4]].values[0][0]).split("  ")[0]
        wei_carry = str(entries1_df.iloc[[h], [4]].values[0][0]).split("  ")[1]
        all_result = str(entries1_df.iloc[[h], [6]].values[0][0]).split("  ")[0]
        graded_result = str(entries1_df.iloc[[h], [6]].values[0][0]).split("  ")[1]
        prize = str(entries1_df.iloc[[h], [6]].values[0][0]).split("  ")[2]
        HrsInfo.append([horse_number, horse_name, sex_age, weight, jockey, wei_carry, all_result, graded_result, prize, ' '])
    #print(HrsInfo[0])

    """ 1-1) 前走のレースと着順 """
    lastRuns = entries2_soup.find_all(class_=re.compile("denmaLs mgnBL"))
    lastRuns = str(lastRuns).split("<td class=\"fntN\">") # 馬ごとのコード
    title_ls =[]
    order_ls =[]
    for i in range(1, len(lastRuns)): # lastRuns[0] 表のヘッダを除く 頭数
        ls = lastRuns[i].split("<div class=") # レースごとのコード
        ls = [re.sub(r"\n","", str(ls[j])) for j in range(1, len(ls))] # ls[0] の馬情報を除く
        """ 前走のレース名 """
        titles = [re.search(r"<strong>(.*?)</strong>", str(ls[j][12:90])).group(1) for j in range(len(ls))]
        titles = titles[::-1] # 反転
        if(len(titles)<5): # データなしNaN
            nan = 5 - len(titles)
            for n in range(nan):
                titles.append('')
        title_ls.append(titles)
        """ 前走の着順 """
        orders = [ls[j][12:14] for j in range(len(ls))]
        orders = orders[::-1] # 反転
        for i in range(len(orders)):
            if(orders[i].isdecimal()):
                orders[i] = orders[i] + "着"
            else:
                orders[i] = "xx着"

        if(len(orders)<5): # データなしNaN
            nan = 5 - len(orders)
            for n in range(nan):
                orders.append('') # NaN
        order_ls.append(orders)

    OdrTitle = []
    for h in range(len(entries2_df)):
        ls = [order_ls[h][r] + " " + title_ls[h][r] for r in range(5)]
        OdrTitle.append(ls)

    NameOdrTitle =[]
    for h in range(len(entries2_df)):
        NameOdrTitle.append(HrsInfo[h] + OdrTitle[h])

    #print(NameOdrTitle[0])

    """ 1-2) 馬体重・騎手 """
    WeightJockey=[]
    for h in range(len(entries2_df)):
        ls = []
        for r in [8, 7, 6, 5, 4]:
            etr2 = str(re.sub(" - ","-", str(entries2_df.iloc[[h], [r]].values[0][0]))) # 馬体重の( - )
            etr2 = re.sub("  ", " ", etr2)
            if(etr2 == 'nan'):
                elm = ''
            elif(len(etr2.split(" ")) < 8):
                elm = etr2.split(" ")[4] # ずれた騎手名（競走除外or中止）
            else:
                elm = etr2.split(" ")[7] + " " + etr2.split(" ")[5]  + " " + etr2.split(" ")[6] # 体重・騎手・斤量
            ls.append(elm)
        WeightJockey.append(ls)

    NameWeiJockey =[]
    for h in range(len(entries2_df)):
        NameWeiJockey.append(HrsInfo[h] + WeightJockey[h])

    #print(NameWeiJockey[0])

    """ 1-3) 過去５走コース・タイム """
    #print(NameWeiJockey[0])
    CourseTime= []
    for h in range(len(entries2_df)):
        ls =[]
        for r in [8, 7, 6, 5, 4]:
            etr2 = str(re.sub(" - ","-", str(entries2_df.iloc[[h], [r]].values[0][0]))) # 馬体重の( - )
            etr2 = re.sub("  ", " ", etr2)
            if(etr2 == 'nan'):
                elm = ''
            elif(len(etr2.split(" ")) < 8):
                elm = etr2.split(" ")[2] + " " + etr2.split(" ")[3] # 競走除外or中止
            else:
                elm = etr2.split(" ")[2] + " " + etr2.split(" ")[3] + " " + etr2.split(" ")[4] # exsample 2/3(中京) 芝2000  2.02.2 
            ls.append(elm)
        CourseTime.append(ls)

    NameCrsTime =[]
    for hrs in range(len(entries3_df)):
        NameCrsTime.append(HrsInfo[hrs] + CourseTime[hrs])

    #print(NameCrsTime[0])

    """ 1-4) 過去５走コーナー順位・人気 """
    CnrPopularityOdr=[]
    for h in range(len(entries3_df)):
        ls =[]
        for r in [8, 7, 6, 5, 4]:
            etr3 = str(re.sub("  "," ", str(entries3_df.iloc[[h], [r]].values[0][0])))
            if(etr3 == 'nan'):
                elm = ''
            elif(len(etr3.split(" ")) < 4):
                elm = "競走除外or中止"
            else:
                elm = etr3.split(" ")[5] + " " + etr3.split(" ")[6]
            ls.append(elm) # 
        CnrPopularityOdr.append(ls)

    NameCnrPopOdr =[]
    for h in range(len(entries3_df)):
        NameCnrPopOdr.append(HrsInfo[h] + CnrPopularityOdr[h])

    #print(NameCnrPopOdr[0])
    
    MyEntries =[]
    for h in range(len(entries2_df)):
        MyEntries.append(NameOdrTitle[h])
        MyEntries.append(NameCrsTime[h])
        MyEntries.append(NameWeiJockey[h])
        MyEntries.append(NameCnrPopOdr[h])

    set_df = pd.DataFrame(MyEntries, columns=['馬番', '馬名', 'S歳', '体重', '騎手', '斤量', '全成績', '重賞成績', '総賞金', '-', \
                                            '前走', '前々走', '3走前', '4走前', '5走前']) # index=idx_horse, 

    myEntries_df = set_df.set_index(['馬番','馬名', 'S歳', '体重', '騎手', '斤量', '全成績', '重賞成績', '総賞金', '-'])
    #myEntries_df.sort_index(level=0)
    return info_race, myEntries_df
    MyEntries =[]
    for h in mySortList:
        MyEntries.append(NameOdrTitle[h])
        MyEntries.append(NameCrsTime[h])
        MyEntries.append(NameWeiJockey[h])
        MyEntries.append(NameCnrPopOdr[h])

    set_df = pd.DataFrame(MyEntries, columns=['馬番', '馬名', 'S歳', '体重', '騎手', '斤量', '全成績','重賞成績', '総賞金', \
                                              '-', '前走', '前々走', '3走前', '4走前', '5走前']) # index=idx_horse, 

    myEntries_df = set_df.set_index(['馬番','馬名', 'S歳', '体重', '騎手', '斤量', '全成績', '重賞成績', '総賞金', '-'])
    #myEntries_df.sort_index(level=0)
    return myEntries_df

""" 人気・血統・コメント """
def myPopularityDF( race_code):
    """ オッズ """
    odds_url="https://race.netkeiba.com/?pid=race_old&id=c2019" + race_code
    sleep(1)
    od_df=pd.io.html.read_html(odds_url)

    od_df[0].columns = od_df[0].columns.droplevel() # 2重のカラム名 MultiIndex の1行削除
    odds_df = od_df[0].drop("馬体重", axis=1)

    """ 調教師コメント """
    train_url="https://race.netkeiba.com/?pid=race_old&id=c2019" + race_code + "&mode=oikiri"
    sleep(1)
    cmt_df=pd.io.html.read_html(train_url)
    cdf = cmt_df[0]
    comment_df = pd.concat([DataFrame(cdf['評価.1']), DataFrame(cdf['評価'])], axis=1)

    """ 血統 ( from 出馬表1) """
    entries1_html = urllib.request.urlopen("https://keiba.yahoo.co.jp/race/denma/19" + race_code + "/?page=1")
    entries1_soup = BeautifulSoup(entries1_html,"lxml")
    df1 = pd.io.html.read_html(entries1_soup.prettify())
    entries1_df = df1[2]
    
    blood = []
    for hrs in range(len(entries1_df)):
        bld = str(entries1_df.iloc[[hrs], [5]].values[0][0]).replace("  ", "/")
        bld = bld.replace("/(", "(")
        blood.append(bld)

    blood_df = pd.DataFrame(blood, columns=['血統'])
    #print(blood_df)
    popularity_df = pd.concat([odds_df, blood_df, comment_df], axis=1)
    
    return popularity_df

print("code loading ", datetime.now().strftime("%Y/%m/%d %H:%M:%S"))

code loading  2019/09/13 21:05:06


In [20]:
#race_code = rc["阪神sat"] + "03"
race_code = rc["中山sun"] + "09"
race_meta, race_df = myEntriesDF(race_code)
print(race_meta[:5], '\n', race_meta[5:11])
display(race_df)

['2019年9月15日（日）', '4回中山4日', '9R', '汐留特別', '14:35発走'] 
 ['芝・右2000m', '天気：晴', '馬場：良', 'サラ系3歳以上', '1勝クラス(500万下)指定定量', '本賞金：1050、420、260、160、105万円']


前走  \
馬番 馬名        S歳  体重     騎手     斤量   全成績      重賞成績    総賞金   -                          
01 マイネルエキサイト 牡3  -( - ) 柴田 大知  54.0 1.1.0.7  0.0.0.0 780万              09着 山藤賞(500)   
                                                              4/13(中山) 芝1800 1.49.8   
                                                                  472(-6) 柴田大知 (56)   
                                                                   35.7 01-01-01-01   
02 フィデリオグリーン 牡3  -( - ) 野中 悠太郎 54.0 1.0.2.5  0.0.0.1 1019万           03着 雄国沼(1勝クラス)   
                                                              7/13(福島) 芝1800 1.48.5   
                                                                 494(-4) 野中悠太郎 (54)   
                                                                   34.3 08-08-08-09   
03 ブラッドストーン  牡3  -( - ) 吉田 豊   54.0 1.1.2.2  0.0.0.0 1252万           03着 3歳上(1勝クラス)   
                                                              6/29(福島) 芝2600 2.40.6   
                                                                   474(+2) 吉田豊 (53)   
                                                                   36.9 02-02-02-02   
04 ウィナーポイント  牝4  -( - ) 横山 典弘  55.0 1.0.4.10 0.0.0.0 1450万           03着 糸魚川(1勝クラス)   
                                                              8/31(新潟) 芝2000 2.00.0   
                                                                  418(+4) 田辺裕信 (55)   
                                                                         34.3 11-11   
05 ウォルフズハウル  牡3  -( - ) 石橋 脩   54.0 1.1.0.1  0.0.0.0 700万                   01着 未勝利   
                                                               7/7(福島) 芝2000 2.03.1   
                                                                   462(+6) 石橋脩 (56)   
                                                                   36.5 01-01-01-01   
06 ナミブ       牡3  -( - ) 松岡 正海  54.0 1.0.2.17 0.0.0.0 1010万           07着 3歳上(1勝クラス)   
                                                              8/31(札幌) 芝2000 2.04.3   
                                                                 452(+12) 松岡正海 (54)   
                                                                   37.6 07-07-07-09   
07 ダイシンクイント  牡5  -( - ) 田中 勝春  57.0 1.1.2.18 0.0.0.0 1818万           05着 糸魚川(1勝クラス)   
                                                              8/31(新潟) 芝2000 2.00.1   
                                                                 474(-10) 田中勝春 (57)   
                                                                         36.6 02-02   
08 ツクバソヴァール  牡3  -( - ) 大野 拓弥  54.0 1.0.4.8  0.0.0.0 1512万           03着 3歳上(1勝クラス)   
                                                              6/23(東京) 芝2000 2.00.5   
                                                                 470(+10) 松岡正海 (54)   
                                                                      36.2 04-04-05   
09 クリノオスマン   せん5 -( - ) 宮崎 北斗  57.0 0.1.2.8  0.0.0.0 996万            02着 糸魚川(1勝クラス)   
                                                              8/31(新潟) 芝2000 1.59.8   
                                                                   450(0) 宮崎北斗 (57)   
                                                                         34.5 05-04   

                                                                                前々走  \
馬番 馬名        S歳  体重     騎手     斤量   全成績      重賞成績    総賞金   -                          
01 マイネルエキサイト 牡3  -( - ) 柴田 大知  54.0 1.1.0.7  0.0.0.0 780万               06着 3歳(500)   
                                                              3/10(中山) 芝2000 2.02.5   
                                                                  478(+2) 柴田大知 (56)   
                                                                   37.1 01-02-02-02   
02 フィデリオグリーン 牡3  -( - ) 野中 悠太郎 54.0 1.0.2.5  0.0.0.1 1019万             12着 スプS(GII)   
                                                              3/17(中山) 芝1800 1.48.8   
                                                                 498(-4) 野中悠太郎

In [21]:
""" 人気情報(単オッズ) """
PopularityDF = myPopularityDF( race_code)
print(race_meta[:6])
PopularityDF.sort_values(by='人気')

['2019年9月15日（日）', '4回中山4日', '9R', '汐留特別', '14:35発走', '芝・右2000m']


,枠番,馬番,馬名,性齢,負担重量,騎手,厩舎,単勝オッズ,人気,血統,評価.1,評価
2,3,3,ブラッドストーン,牡3,54.0,吉田豊,小島,3.3,1,ローズキングダム/マレンカヤ(ファンタスティックライト),C,仕上十分
4,5,5,ウォルフズハウル,牡3,54.0,石橋脩,栗田,3.8,2,ハーツクライ/アークティックシルク(Selkirk),B,伸び上々
3,4,4,ウィナーポイント,牝4,55.0,横山典,和田勇,5.9,3,スクリーンヒーロー/メイクデュース(トウカイテイオー),B,状態良好
6,7,7,ダイシンクイント,牡5,57.0,田中勝,森,8.6,4,ルーラーシップ/ダイシンキャンディ(アグネスタキオン),B,叩き良化
8,8,9,クリノオスマン,セ5,57.0,宮崎,伊藤伸,8.7,5,ナカヤマフェスタ/ダービーゾーン(サニーブライアン),C,先着平凡
7,8,8,ツクバソヴァール,牡3,54.0,大野,杉浦,8.9,6,ロジユニヴァース/リリーオブザナイル(バゴ),C,前走並み
1,2,2,フィデリオグリーン,牡3,54.0,野中,根本,11.3,7,ザサンデーフサイチ/ミサノグリーン(タニノギムレット),C,まずまず
0,1,1,マイネルエキサイト,牡3,54.0,柴田大,加藤士,20.6,8,ブラックタイド/アインライツ(ティンバーカントリー),C,遅れ上々
5,6,6,ナミブ,牡3,54.0,松岡,南田,22.0,9,ナカヤマフェスタ/マイネジャンヌ(ロージズインメイ),B,動き良化


In [93]:
""" 結果 """
result_html =  "https://keiba.yahoo.co.jp/race/result/19" + race_code + "/"
ResultDF = pd.io.html.read_html(result_html)
print(race_meta[:5])
ResultDF[4]

['2019年8月25日（日）', '2回札幌4日', '11R', 'キーンランドカップ（GIII）', '15:35発走']


,着順,枠番,馬番,馬名性齢⁄馬体重⁄B,タイム(着差),通過順位上3Fタイム,騎手,人気(オッズ),調教師
0,1,7,13,ダノンスマッシュ 牡4/478(+6)/,1.09.2,07-0635.3,川田 将雅 57.0,1 (2.3),安田 隆行
1,2,4,7,タワーオブロンドン 牡4/520(+2)/,1.09.33/4馬身,12-1234.9,C.ルメール 58.0,2 (4.7),藤沢 和雄
2,3,8,16,リナーテ 牝5/488(-8)/,1.09.3ハナ,08-0835.2,武 豊 54.0,3 (6.6),須貝 尚介
3,4,7,14,ライトオンキュー 牡4/514(-2)/,1.09.4クビ,12-1135.1,古川 吉洋 56.0,11 (42.5),昆 貢
4,5,1,1,ナックビーナス 牝6/526(+3)/,1.09.61 1/2馬身,01-0136.4,岩田 康誠 55.0,4 (12.6),杉浦 宏昭
5,6,2,4,セイウンコウセイ 牡6/502(-4)/,1.09.73/4馬身,03-0436.3,幸 英明 58.0,5 (13.8),上原 博之
6,7,3,5,ペイシャフェリシタ 牝6/494(0)/,1.09.8クビ,04-0236.2,K.ティータン 54.0,13 (63.1),高木 登
7,8,1,2,デアレガーロ 牝5/482(+2)/,1.09.8クビ,08-0835.7,池添 謙一 54.0,6 (15.6),大竹 正博
8,9,8,15,パラダイスガーデン 牝7/502(+4)/,1.09.91/2馬身,16-1634.9,横山 武史 54.0,15 (223.9),粕谷 昌央
9,10,3,6,ハッピーアワー 牡3/446(+4)/,1.10.03/4馬身,15-1435.4,吉田 隼人 54.0,8 (21.1),武 幸四郎


In [92]:
""" 払い戻し """
print(race_meta[:5])
tst_df = pd.concat([ResultDF[2], ResultDF[3]])
tst_df.reset_index(drop=True)

['2019年8月25日（日）', '2回札幌4日', '11R', 'キーンランドカップ（GIII）', '15:35発走']


,0,1,2,3
0,単勝,13,230円,1番人気
1,複勝,13,110円,1番人気
2,複勝,7,150円,2番人気
3,複勝,16,180円,3番人気
4,枠連,4－7,640円,1番人気
5,馬連,7－13,760円,1番人気
6,ワイド,7－13,300円,1番人気
7,ワイド,7－16,640円,5番人気
8,ワイド,13－16,330円,2番人気
9,馬単,13－7,"1,170円",1番人気


In [5]:
""" my出馬表の人気順・結果順 """
pS_ls = [PopularityDF.sort_values(by='人気').iloc[[h],[1]].values.tolist() for h in range(len(entries1_df))]
popSort = [pS_ls[h][0][0] for h in range(len(entries1_df))]
PopulartySort = [ h - 1 for h in popSort]

""" my出馬表の結果順 """
rS_ls = [ResultDF[4].iloc[[h],[2]].values.tolist() for h in range(len(entries1_df))]
rltSort = [rS_ls[h][0][0] for h in range(len(entries1_df))]
ResultSort = [ h - 1 for h in rltSort]

print(re.sub(r"[\[',\]]","", str(info_basic)), " ", str(info_meta[0]), str(info_meta[1]), str(info_meta[2]))
# myEntriesDF(PopulartySort)
myEntriesDF(ResultSort)

NameError: name 'entries1_df' is not defined

In [333]:
url="https://race.netkeiba.com/?pid=race_old&id=c201901020301"
print(url)

https://race.netkeiba.com/?pid=race_old&id=c201901020301


In [223]:
for s in crs_dic.keys():
    if(s[2:]=='sat'): w = '(土)'
    else: w = '(日)'
    print(s[:2], w)
    print(crs_dic[s])

札幌 (土)
010205
新潟 (土)
040211
小倉 (土)
100211
札幌 (日)
010206
新潟 (日)
040212
小倉 (日)
100212


In [3]:
import unicodedata
def left(digit, msg):
    for c in msg:
        if unicodedata.east_asian_width(c) in ('F', 'W', 'A'):
            digit -= 2
        else:
            digit -= 1
    return msg + ' '*digit

""" 掲示板内の結果 """
rc = {}
rc['札幌sat'] = '010205' #8/31
rc['新潟sat'] = '040211'
rc['小倉sat'] = '100211'
rc['札幌sun'] = '010206' #9/1
rc['新潟sun'] = '040212'
rc['小倉sun'] = '100212'

for v in rc.values():
    for r in range(12):
        result_html =  "https://keiba.yahoo.co.jp/race/result/19" + str(v) + str(r+1).rjust(2,"0")
        sleep(1)
        ResultDF = pd.io.html.read_html(result_html)
        nnR = ResultDF[1].iloc[[0]].values[0][0]
        rcode = ResultDF[1].iloc[[0]].values[0][1].split(" | ")[1]
        rtitle = ResultDF[1].iloc[[0]].values[0][1].split(" | ")[2]
        print(rcode, nnR, rtitle)
        """ レース結果 """
        for i in range(8):
            rslt = ResultDF[4].iloc[[i]]
            odr = str(rslt.iloc[[0], [2]].values[0][0])
            hrs = str(rslt.iloc[[0], [3]].values[0][0]).split("  ")[0]
            pop = rslt.iloc[[0], [7]].values[0][0]
            print("{:} {:} {:}".format( odr.rjust(2), left(25, hrs) , pop))

2回札幌5日 1R 9:50発走 サラ系2歳未勝利 芝・右 2000m
 2 ミヤマザクラ              2 (2.6)
14 ナスノフォルテ            9 (135)
 9 メリディアンローグ        1 (1.9)
10 アベルゴー                10 (225.5)
 3 エピファレーヌ            3 (11)
13 アピテソーロ              5 (25.7)
 8 シャーベットフィズ        16 (605.5)
 7 サイモンルグラン          8 (99.8)
2回札幌5日 2R 10:25発走 サラ系2歳未勝利 ダート・右 1700m
12 デルマオニキス            9 (39.9)
 7 クレパト                  3 (11.1)
 2 バカラ                    1 (1.3)
14 ディーエスプルーフ        6 (25.2)
 6 ウインアクティーボ        2 (8)
 1 ヤマノマタカ              5 (20.1)
 5 ラフリッグフェル          4 (17.1)
 9 ワンダーカムラング        8 (39.7)
2回札幌5日 3R 10:55発走 サラ系3歳未勝利 ダート・右 1000m
12 ファロ                    2 (3.8)
11 テイエムダイリン          4 (6.2)
 4 グランデオーロラ          3 (6.2)
 3 モエレコネクター          8 (22.2)
 1 エルマスフエルテ          9 (27.5)
 9 ビービーアライヴ          1 (3)
10 タカラチーター            11 (115.2)
 5 ローマノキュウジツ        7 (20)
2回札幌5日 4R 11:25発走 サラ系3歳未勝利 芝・右 1200m
 6 レッドクレオス            1 (1.9)
15 ドゥシャンパーニュ        4 (12.5)
10 ソルパシオン              5 (12.7)
16 マイネルカゲツ            10 (27

2回小倉11日 6R 12:55発走 サラ系3歳未勝利 ダート・右 1700m
10 アイタイ                  1 (3.3)
 2 サンライズアカシア        5 (8)
 3 スズノチェルシー          8 (23.3)
 1 メイショウミチノク        4 (7)
 4 フェブタイズ              10 (30.8)
16 オウケンシャトル          16 (368.4)
11 メルテール                7 (18.4)
 9 スーパーアロイ            2 (5.2)
2回小倉11日 7R 13:25発走 サラ系3歳未勝利 芝・右 2000m
 4 ルベリエ                  3 (6.5)
14 リッカローズ              5 (9.3)
18 ブルーエクセレンス        1 (3)
 6 レッドジェニファー        7 (21.4)
13 レッドレイル              6 (19.9)
 2 アドマイヤジョイ          9 (25.7)
 3 エバークリア              13 (50)
 9 スノーユニバンス          2 (4.9)
2回小倉11日 8R 13:55発走 サラ系3歳上1勝クラス(500万円以下) ダート・右 1000m
 7 ノボリレーヴ              1 (3.2)
 4 サイモンゼーレ            7 (12)
 3 アユツリオヤジ            6 (10.2)
 8 トーホウレジーナ          2 (4.6)
 9 シルバークレイン          11 (37.9)
 1 ブライトメジャー          9 (23.8)
10 レナータ                  10 (24.5)
12 アメリカンソレイユ        5 (8.8)
2回小倉11日 9R 14:25発走 八幡特別 芝・右 1200m
12 キャスパリーグ            2 (4.1)
15 ブルベアオーロ            4 (6.6)
 1 クーファディーヴァ        3 (5.6)
 9 メイショウナスカ         

2回新潟12日 11R 15:45発走 第55回農林水産省賞典 新潟記念（GIII） 芝・左・外 2000m
 7 ユーキャンスマイル        2 (6.3)
 5 ジナンボー                6 (11.1)
 6 カデナ                    8 (14.2)
15 ブラックスピネル          11 (32)
 4 フランツ                  3 (8.6)
12 ショウナンバッハ          14 (65.3)
16 センテリュオ              4 (9.6)
14 サトノワルキューレ        12 (33.9)
2回新潟12日 12R 16:30発走 雷光特別 芝・直線 1000m
16 ピカピカ                  4 (6.7)
15 エムオールビー            5 (6.7)
18 セイウンアカマイ          2 (4.8)
17 ウインアイルビータ        3 (4.9)
12 ヤマニンベリンダ          9 (26.3)
 6 ヤマニンベルベーヌ        11 (68.1)
10 グラスレガシー            6 (10.6)
13 テンモントム              10 (49.3)
2回小倉12日 1R 10:01発走 サラ系2歳未勝利 ダート・右 1700m
15 キメラヴェリテ            6 (14.7)
 8 スマートラビリンス        5 (10.6)
 2 ジョウショーリード        13 (192.5)
 4 エコロブラッサム          3 (5.9)
 9 ジョーコレット            9 (40.3)
 5 サンビースト              2 (5.6)
14 クリノヴジュアル          10 (41.7)
 6 タガノカスターニャ        4 (6.2)
2回小倉12日 2R 10:35発走 サラ系2歳未勝利 芝・右 1200m
 7 エグレムニ                1 (2.4)
12 カリオストロ              2 (4.2)
 6 アビエルト                7 (17)
 5

In [22]:
1-(49/50)**51

0.6431137135146257